In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("data.csv", encoding="latin1")

print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  


In [204]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [205]:
# Checking for null values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

# 1. Fill missing Descriptions



In [206]:
# Check how many empty descriptions exist
print(df["Description"].isnull().sum())

1454


In [207]:
df["Description"] = df["Description"].fillna("Unknown")

# 2. Remove rows with missing CustomerID

In [208]:
# Check count of missing IDs
print(df["CustomerID"].isnull().sum())

135080


In [209]:
df = df.dropna(subset=["CustomerID"])

# 3. Remove the invalid StockCodes

In [210]:
# Define the invalid StockCodes
invalid_stockcodes = [
    "POST",
    "C2",
    "M",
    "DOT",
    "BANK CHARGES"
]

#Remove rows containing these StockCodes
df = df[~df["StockCode"].isin(invalid_stockcodes)]

In [211]:
#Check that no invalid StockCodes remain
df[df["StockCode"].isin(invalid_stockcodes)]

#Check how many rows remain
print(f"Rows remaining after removing invalid StockCodes: {len(df)}")

Rows remaining after removing invalid StockCodes: 405006


# 4. Remove Duplicates

In [212]:
# Count duplicate rows
print(f"Duplicates found: {df.duplicated().sum()}")

Duplicates found: 5220


In [213]:
df = df.drop_duplicates()

# 5. Convert InvoiceDate to datetime

In [4]:
# Check the data type
print(df["InvoiceDate"].dtype)
print(df["InvoiceDate"].head(3))

object
0    12/1/2010 8:26
1    12/1/2010 8:26
2    12/1/2010 8:26
Name: InvoiceDate, dtype: object


In [5]:
# Convert to datetime
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Remove time and keep only date
df["InvoiceDate"] = df["InvoiceDate"].dt.date

print(df["InvoiceDate"].head(3))

0    2010-12-01
1    2010-12-01
2    2010-12-01
Name: InvoiceDate, dtype: object


# 6. Create TotalSales Column

In [216]:
# Preview the inputs
print(df[["Quantity", "UnitPrice"]].head())

   Quantity  UnitPrice
0         6       2.55
1         6       3.39
2         8       2.75
3         6       3.39
4         6       3.39


In [217]:
df["TotalSales"] = df["Quantity"] * df["UnitPrice"]

# 7. Round TotalSales to 2 decimals

In [218]:
# Check the un-rounded numbers
print(df["TotalSales"].head())

0    15.30
1    20.34
2    22.00
3    20.34
4    20.34
Name: TotalSales, dtype: float64


In [219]:
df["TotalSales"] = df["TotalSales"].round(2)


# 8. Remove Cancelled Transactions

In [220]:
# Show me rows where InvoiceNo starts with 'C'
cancelled = df[df["InvoiceNo"].astype(str).str.startswith("C")]
print(f"Cancelled transactions found: {len(cancelled)}")
print(cancelled.head(3))

Cancelled transactions found: 8599
    InvoiceNo StockCode                      Description  Quantity  \
141   C536379         D                         Discount        -1   
154   C536383    35004C  SET OF 3 COLOURED  FLYING DUCKS        -1   
235   C536391     22556   PLASTERS IN TIN CIRCUS PARADE        -12   

    InvoiceDate  UnitPrice  CustomerID         Country  TotalSales  
141  2010-12-01      27.50     14527.0  United Kingdom      -27.50  
154  2010-12-01       4.65     15311.0  United Kingdom       -4.65  
235  2010-12-01       1.65     17548.0  United Kingdom      -19.80  


In [221]:
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# 9. Remove Invalid Values

In [222]:
# Show me rows where Quantity is 0 or less, OR UnitPrice is 0 or less
bad_data = df[(df["Quantity"] <= 0) | (df["UnitPrice"] <= 0)]
print(f"Invalid rows found: {len(bad_data)}")
bad_data.head(34)

Invalid rows found: 34


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSales
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05,0.0,12647.0,Germany,0.0
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16,0.0,16560.0,United Kingdom,0.0
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21,0.0,14911.0,EIRE,0.0
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06,0.0,13081.0,United Kingdom,0.0
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06,0.0,13081.0,United Kingdom,0.0
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13,0.0,15107.0,United Kingdom,0.0
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10,0.0,17560.0,United Kingdom,0.0
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23,0.0,13239.0,United Kingdom,0.0
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30,0.0,13113.0,United Kingdom,0.0
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04,0.0,14410.0,United Kingdom,0.0


In [223]:
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# 10. Trim Extra Spaces

In [224]:
# Check a specific string column
print(df["Description"].unique()[0:5])

['WHITE HANGING HEART T-LIGHT HOLDER' 'WHITE METAL LANTERN'
 'CREAM CUPID HEARTS COAT HANGER' 'KNITTED UNION FLAG HOT WATER BOTTLE'
 'RED WOOLLY HOTTIE WHITE HEART.']


In [1]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

NameError: name 'df' is not defined

## Checking the structure of the data

In [ ]:
df.info()

## Importing the dataset

In [231]:
import pandas as pd

df = pd.read_csv("Cleaned_Data.csv", encoding="latin1")

# Pass 1: Day-first (UK style)
dates_uk = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce",
    dayfirst=True
)

# Pass 2: Month-first (US style)
dates_us = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce",
    dayfirst=False
)

# Combine both: use UK first, fallback to US
df["InvoiceDate_Clean"] = dates_uk.fillna(dates_us)

# Check how many are still NaT
print("Unparsed dates:", df["InvoiceDate_Clean"].isna().sum())

# Optional: keep date only (remove time)
df["InvoiceDate_Clean"] = df["InvoiceDate_Clean"].dt.date

# ✅ IMPORTANT: Do NOT drop rows blindly
# Drop only rows where BOTH parses failed
df = df.dropna(subset=["InvoiceDate_Clean"])

# Optional: replace old column
df = df.drop(columns=["InvoiceDate"])
df = df.rename(columns={"InvoiceDate_Clean": "InvoiceDate"})

# Save cleaned file
df.to_csv("Cleaned_Data_PBI.csv", index=False)

Unparsed dates: 0


In [232]:
print(df["InvoiceDate"].dtype)
print(df["InvoiceDate"].head())
print(df["InvoiceDate"].isna().sum())

object
0    2010-01-12
1    2010-01-12
2    2010-01-12
3    2010-01-12
4    2010-01-12
Name: InvoiceDate, dtype: object
0


In [233]:
print("Rows before:", len(pd.read_csv("Cleaned_Data.csv")))
print("Rows after:", len(df))

Rows before: 391153
Rows after: 391153
